In [1]:
import os
from tqdm.auto import tqdm
import pandas as pd

# to use data structures
from pydantic import BaseModel, ConfigDict
# from enum import Enum

# OpenAI API
from openai import OpenAI


# Load the API keys from .env
from dotenv import load_dotenv

pd.options.mode.chained_assignment = (
    None  # default='warn' This disables warning of "copying a slice of a DataFrame"
)
tqdm.pandas()  # activate the tqdm for pandas

In [6]:
# load the API keys
load_dotenv(".env")
for api_key in ["OPENAI_API_KEY"]:
    if os.getenv(api_key) is not None:
        print(api_key, "loaded")
    else:
        print(api_key, "missing")
        print("Please create a .env file with the corresponding API key")

OPENAI_API_KEY loaded


In [9]:
from utils.set_paths import (
    path_app_context,
    path_templates,
    path_data,
    path_temp,
    path_output,
)

path_batch_files = path_data / "batches"

# import util functions
from utils.content_utils import *

In [4]:
summary_excel = pd.read_excel("../EconJM_status.xlsx", sheet_name="sum")
summary_excel

,add_id,app_status,institution,location,add_rank,add_field,add_category,deadline_target,add_posting,add_how_apply,needed_cover_letter,needed_RS,needed_TS,OMER_letter_status,KLAUS_letter_status,WOO_letter_status
0,simon_frazer_1,prep docs,Simon Fraser University,"Burnaby, Canada",assistant,applied micro,research,2025-10-01,EJM,EJM,1,1,1,NaN,NaN,NaN
1,macmaster_1,prep docs,McMaster University,"Hamilton, Canada",assistant,applied micro,teaching,2025-10-01,EJM,EJM,1,1,1,NaN,NaN,NaN


# Load the needed input files

- Research Statement Template
- Cover Letter Template
- Teaching Statement Template

In [5]:
# Research Statement template
with open(path_templates / "research_statement/Gonzalez_RS_template.typ", "r") as f:
    lines = f.readlines()

# Find the line where the research statement starts
start_idx = next(
    i for i, line in enumerate(lines) if line.strip().startswith("= Research Statement")
)
research_statement_lines = lines[start_idx:]
RA_TEMPLATE = " ".join(research_statement_lines)

# Figure out what docs are needed to generate

In [27]:
columns_to_show = [
    "add_id",
    "institution",
    "location",
    "add_rank",
    "add_field",
    "add_category",
    "needed_cover_letter",
    "needed_RS",
    "needed_TS",
]

In [ ]:
completed_docs = [d for d in path_output.iterdir() if d.is_dir()]

docs_to_generate = set(summary_excel.add_id.to_list()) - set(completed_docs)

if docs_to_generate:
    print("Need to generate docs for:")
    to_generate_df = summary_excel[summary_excel.add_id.isin(docs_to_generate)][
        columns_to_show
    ]
    docs_list = to_generate_df.astype(str).values.tolist()
    for id in docs_to_generate:
        print(f"- {id}")
else:
    print("No docs needed to generate")

Need to generate docs for:
- macmaster_1
- simon_frazer_1


In [ ]:
to_generate_df.head()

,add_id,institution,location,add_rank,add_field,add_category,needed_cover_letter,needed_RS,needed_TS
0,simon_frazer_1,Simon Fraser University,"Burnaby, Canada",assistant,applied micro,research,1,1,1
1,macmaster_1,McMaster University,"Hamilton, Canada",assistant,applied micro,teaching,1,1,1


# Add the context and additional data for each prompr

In [33]:
def get_txt_info(file_path):
    if not file_path.exists():
        return ""
    else:
        with open(file_path, "r") as f:
            text = f.read()
    return text

In [ ]:
# get the add description context
to_generate_df.loc[:, "add_description"] = to_generate_df.apply(
    lambda row: get_txt_info(
        path_app_context / f"add_descriptions/{row['add_id']}.txt"
    ),
    axis=1,
)

to_generate_df.loc[:, "econ_context"] = to_generate_df.apply(
    lambda row: get_txt_info(path_app_context / f"econ_dept/{row['add_id']}.txt"),
    axis=1,
)

to_generate_df.loc[:, "institution_values"] = to_generate_df.apply(
    lambda row: get_txt_info(
        path_app_context / f"institution_values/{row['add_id']}.txt"
    ),
    axis=1,
)

In [43]:
to_generate_df

,add_id,institution,location,add_rank,add_field,add_category,needed_cover_letter,needed_RS,needed_TS,add_description,econ_context,institution_values
0,simon_frazer_1,Simon Fraser University,"Burnaby, Canada",assistant,applied micro,research,1,1,1,ASSISTANT PROFESSOR OPENING\n\nSimon Fraser Un...,\n\nThe dynamic teaching and learning environm...,\nOur Mission\n\nOur goal is to shift the SFU ...
1,macmaster_1,McMaster University,"Hamilton, Canada",assistant,applied micro,teaching,1,1,1,McMaster University is located on the traditio...,We invite you to browse our website and discov...,"Our Mission\n\nAt McMaster, our purpose is the..."


# OpenAI Batches

In [7]:
openai_client = OpenAI();

In [10]:
# Load the OpenAI batching df
# load history of batches
if (path_batch_files / "batches_history.csv").exists():
    batch_history = pd.read_csv(path_batch_files / "batches_history.csv")
else:
    print("No history of batches found")
    batch_history = pd.DataFrame(
        columns=[
            "created_at",
            "description",
            "model",
            "cat_type",
            "batch_id",
            "status",
            "processing_status",
            "input_file_id",
            "output_file_id",
            "error_file_id",
        ]
    )

batch_history = update_batch_status(batch_history, openai_client)
batch_history.sort_values(["created_at", "model", "cat_type"], inplace=True)
batch_history.to_csv(path_batch_files / "batches_history.csv", index=False)


# Check the inprocess batches
in_process_batches = batch_history[batch_history["status"] == "in_progress"]
print("In-process batches:")
display(in_process_batches.tail(5))

# Check the completed batches
completed_batches = batch_history[
    (batch_history["status"] == "completed")
    & (batch_history["processing_status"] == False)
    & (batch_history["error_file_id"].isna())
]
print("Completed batches:")
display(completed_batches.tail(5))

No history of batches found
In-process batches:


,created_at,description,model,cat_type,batch_id,status,processing_status,input_file_id,output_file_id,error_file_id


Completed batches:


,created_at,description,model,cat_type,batch_id,status,processing_status,input_file_id,output_file_id,error_file_id
